# Worked Example: PSD and FOOOF Reward Contrast

## Goal
Compare reward vs no-reward spectra on feedback epochs using the stable analysis spine (`build_analysis_config` + `run_analysis`). FOOOF details: 11_advanced_utility_interoperability.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from LFPAnalysis import build_analysis_config, load_lfp, run_analysis
from LFPAnalysis.config import LoadConfig

beh = pd.read_csv(Path('../../data/sample_beh.csv'))
epochs = load_lfp(
    LoadConfig(path=Path('../../data/sample_feedback_start-epo.fif'), file_format='mne', preload=True)
)
epochs.metadata = beh[['reward', 'rpe']]
chan = 'racas1-racas2'
reward_epochs = epochs['reward == 1'].copy().pick([chan])
loss_epochs = epochs['reward == 0'].copy().pick([chan])
reward_result = run_analysis(reward_epochs, build_analysis_config(spectral_method='psd', fmin=1.0, fmax=80.0))
loss_result = run_analysis(loss_epochs, build_analysis_config(spectral_method='psd', fmin=1.0, fmax=80.0))
reward_psd = reward_result.spectral['spectrum']
loss_psd = loss_result.spectral['spectrum']
print('reward PSD shape:', reward_psd.get_data().shape)

## Plot PSD contrast

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.semilogy(reward_psd.freqs, reward_psd.get_data().mean(axis=0)[0], label='reward')
ax.semilogy(loss_psd.freqs, loss_psd.get_data().mean(axis=0)[0], label='no reward')
ax.set(xlabel='Frequency (Hz)', ylabel='PSD', title=f'{chan} feedback-locked')
ax.legend()
fig.tight_layout()
plt.show()

## FOOOF via stable analysis spine (subset for speed)

In [ ]:
epochs_sub = epochs.copy().pick([chan])[:10]
fooof_result = run_analysis(
    epochs_sub,
    build_analysis_config(spectral_method='fooof', fooof_range=(1.0, 40.0)),
)
fooof_table = fooof_result.spectral['table']
print(fooof_table.head())

## Next step

Advanced utility interoperability: 11_advanced_utility_interoperability. Next chapter: time-frequency (`09_first_time_frequency`).